# Check one real batch through ECGFounder and the main model

Select **Python (SYMILE-DrFuse)** and run the cells in order. This uses the first
**two training admissions**, with all modalities available. ECGFounder's backbone
loads the official pretrained weights. The new branches, CXR/labs encoders and
fusion layers have not been trained, so the probabilities are only a wiring check.
No training, optimizer update or dataset modification takes place.

## 1. Import the new version

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import torch
from IPython.display import display
from torch.utils.data import DataLoader, Subset

# Works when opened from this notebook folder, the new project, or Research.
candidates = [candidate for parent in (Path.cwd(), *Path.cwd().parents)
              for candidate in (parent, parent / "Pipeline2_ECGFounder")]
PROJECT_DIR = next(path.resolve() for path in candidates
                   if path.name == "Pipeline2_ECGFounder"
                   and (path / "models" / "ecg_founder.py").is_file())
sys.path.insert(0, str(PROJECT_DIR))

# A kernel that already imported Pipeline2 must be restarted to switch versions.
if "models" in sys.modules:
    assert Path(sys.modules["models"].__file__).resolve().parent == PROJECT_DIR / "models", (
        "Restart the kernel: it has already imported another version's models."
    )
if "utils" in sys.modules:
    assert Path(sys.modules["utils"].__file__).resolve().parent == PROJECT_DIR / "utils", (
        "Restart the kernel: it has already imported another version's utils."
    )

from models import MainModel
from utils import TARGETS, SymileMIMICDataset

torch.manual_seed(42)
torch.set_num_threads(2)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = PROJECT_DIR.parent / "symile-mimic" / "symile-mimic-aws"
CHECKPOINT = PROJECT_DIR.parent / "ecgfounder" / "checkpoint" / "12_lead_ECGFounder.pth"
print("Project:", PROJECT_DIR)
print("Device:", DEVICE)
print("Pretrained weights:", CHECKPOINT)

## 2. Load two real admissions
The loader applies ECGFounder preprocessing to the stored SYMILE ECGs.

In [ ]:
BATCH_SIZE = 2
train_dataset = SymileMIMICDataset(DATA_DIR, split="train")
loader = DataLoader(Subset(train_dataset, range(BATCH_SIZE)),
                    batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
batch = next(iter(loader))

print("Training admissions:", len(train_dataset))
display(train_dataset.metadata.iloc[:BATCH_SIZE][["subject_id", "hadm_id", *TARGETS]])
display(pd.DataFrame([
    {"batch_item": name, "shape": tuple(value.shape), "dtype": str(value.dtype)}
    for name, value in batch.items()
]))
assert batch["ecg"].shape == (BATCH_SIZE, 12, 5000)
assert batch["labs"].shape == (BATCH_SIZE, 100)
assert torch.isfinite(batch["ecg"]).all()

## 3. Build the model
Only the ECGFounder backbone starts with pretrained weights.

In [ ]:
model = MainModel(
    hidden_size=256,
    num_classes=len(TARGETS),
    ecg_checkpoint=CHECKPOINT,
).to(DEVICE)
model.eval()
print("Model ready for inference.")

## 4. Inspect the intermediate representations

Temporary hooks capture the ECG backbone embedding and both ECG branches during
one complete forward pass. The main model then projects shared representations,
fuses all three modalities and applies disease-wise attention.

In [ ]:
captured = {}

def capture_embedding(module, inputs, output):
    captured["ecg_embedding"] = output.detach()

def capture_ecg_branches(module, inputs, output):
    shared, distinct, predictions = output
    captured["ecg_shared"] = shared.detach()
    captured["ecg_distinct"] = distinct.detach()
    captured["ecg_predictions"] = predictions.detach()

hooks = [model.ecg_model.backbone.register_forward_hook(capture_embedding),
         model.ecg_model.register_forward_hook(capture_ecg_branches)]
try:
    with torch.inference_mode():
        output = model(**{name: batch[name].to(DEVICE)
                          for name in ("cxr", "ecg", "labs")})
finally:
    for hook in hooks:
        hook.remove()

shape_rows = [("Preprocessed ECG input", batch["ecg"]),
              ("ECGFounder embedding", captured["ecg_embedding"]),
              ("ECG shared branch", captured["ecg_shared"]),
              ("ECG distinct branch", captured["ecg_distinct"]),
              ("Fused shared representation", output["feat_avg_shared"]),
              ("Disease-specific representations", output["feat_final"]),
              ("Attention weights", output["attn_weights"]),
              ("Final predictions", output["pred_final"])]
display(pd.DataFrame([{"stage": name, "shape": tuple(value.shape)}
                      for name, value in shape_rows]))
print("Attention tokens:", output["attn_token_names"])

assert captured["ecg_embedding"].shape == (BATCH_SIZE, 1024)
assert captured["ecg_shared"].shape == (BATCH_SIZE, 256)
assert captured["ecg_distinct"].shape == (BATCH_SIZE, 256)
assert output["pred_final"].shape == (BATCH_SIZE, 14)
assert output["attn_weights"].shape == (BATCH_SIZE, 14, 4)
assert all(torch.isfinite(value).all() for _, value in shape_rows)
torch.testing.assert_close(output["attn_weights"].sum(-1),
                           torch.ones(BATCH_SIZE, 14, device=DEVICE))
print("All shape and finite-value checks passed.")

## 5. Show the 14 final probabilities
These are untrained task-head outputs, not evaluation scores.

In [ ]:
probabilities = pd.DataFrame(output["pred_final"].cpu().numpy(), columns=TARGETS)
probabilities.insert(0, "hadm_id", batch["hadm_id"].numpy())
display(probabilities.round(4))

# Known labels are 0/1; NaN and -1 are excluded by this mask.
known_labels = pd.DataFrame(batch["label_mask"].numpy(), columns=TARGETS)
known_labels.insert(0, "hadm_id", batch["hadm_id"].numpy())
print("Valid-label mask for the same admissions:")
display(known_labels)